In [ ]:
import requests
import os
import xarray as xr
import numpy as np
from datetime import datetime

CACHE_DIR = "argo_cache"
BASE_URL  = "https://data-argo.ifremer.fr/geo/indian_ocean/"
os.makedirs(CACHE_DIR, exist_ok=True)

# Sample dates — har saal 4 seasons
sample_dates = []

years = [2021, 2022, 2023, 2024, 2025]
for year in years:
    sample_dates.extend([
        datetime(year, 1,  15),   # Winter
        datetime(year, 4,  15),   # Spring
        datetime(year, 7,  15),   # Summer
        datetime(year, 10, 15),   # Autumn
    ])

# 2026 — Abhi tak ka
sample_dates.extend([
    datetime(2026, 1, 15),
    datetime(2026, 3, 18),  # today
])

print(f"Total dates: {len(sample_dates)}")
print("Downloading...\n")

success = 0
failed  = 0

for date in sample_dates:
    filename   = date.strftime("%Y%m%d") + "_prof.nc"
    year       = date.strftime("%Y")
    month      = date.strftime("%m")
    cache_path = os.path.join(CACHE_DIR, filename)

    # Already hai? Skip!
    if os.path.exists(cache_path):
        print(f"⏭️  Already exists: {filename}")
        success += 1
        continue

    url = f"{BASE_URL}{year}/{month}/{filename}"

    try:
        r = requests.get(url, timeout=120, stream=True)
        if r.status_code == 200:
            with open(cache_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    f.write(chunk)
            size = os.path.getsize(cache_path) / (1024*1024)
            print(f"✅ {filename} — {size:.1f} MB")
            success += 1
        else:
            print(f"❌ {filename} — Status {r.status_code}")
            failed += 1
    except Exception as e:
        print(f"⚠️  {filename} — Error: {e}")
        failed += 1

# Summary
total_size = sum(
    os.path.getsize(os.path.join(CACHE_DIR, f)) 
    for f in os.listdir(CACHE_DIR) 
    if f.endswith('.nc')
) / (1024*1024)

print(f"""
🎉 Download Complete!
━━━━━━━━━━━━━━━━━━━━
✅ Success : {success} files
❌ Failed  : {failed} files
💾 Total   : {total_size:.1f} MB
""")

Total dates: 22
Downloading...

✅ 20210115_prof.nc — 3.9 MB
✅ 20210415_prof.nc — 3.6 MB
✅ 20210715_prof.nc — 4.1 MB
✅ 20211015_prof.nc — 4.8 MB
✅ 20220115_prof.nc — 4.7 MB
✅ 20220415_prof.nc — 4.0 MB
✅ 20220715_prof.nc — 4.1 MB
✅ 20221015_prof.nc — 6.1 MB
✅ 20230115_prof.nc — 4.4 MB
✅ 20230415_prof.nc — 6.0 MB
✅ 20230715_prof.nc — 2.9 MB
✅ 20231015_prof.nc — 3.5 MB
✅ 20240115_prof.nc — 3.9 MB
✅ 20240415_prof.nc — 4.1 MB
✅ 20240715_prof.nc — 5.8 MB
✅ 20241015_prof.nc — 6.2 MB
✅ 20250115_prof.nc — 5.4 MB
✅ 20250415_prof.nc — 5.7 MB
✅ 20250715_prof.nc — 5.0 MB
✅ 20251015_prof.nc — 5.0 MB
✅ 20260115_prof.nc — 9.9 MB
✅ 20260318_prof.nc — 2.7 MB

🎉 Download Complete!
━━━━━━━━━━━━━━━━━━━━
✅ Success : 22 files
❌ Failed  : 0 files
💾 Total   : 110.7 MB



In [5]:
import xarray as xr
import numpy as np

ds = xr.open_dataset("argo_cache/20230115_prof.nc")

print("📊 Important Variables:")
important = ['PLATFORM_NUMBER', 'JULD', 'LATITUDE', 
             'LONGITUDE', 'PRES', 'TEMP', 'PSAL',
             'POSITIONING_SYSTEM', 'PLATFORM_TYPE',
             'DATA_CENTRE', 'PROJECT_NAME']

for var in important:
    if var in ds:
        print(f"  ✅ {var:25s} : {ds[var].dtype} | shape: {ds[var].shape}")
    else:
        print(f"  ❌ {var:25s} : NOT FOUND")
print(ds)

📊 Important Variables:
  ✅ PLATFORM_NUMBER           : object | shape: (74,)
  ✅ JULD                      : datetime64[ns] | shape: (74,)
  ✅ LATITUDE                  : float64 | shape: (74,)
  ✅ LONGITUDE                 : float64 | shape: (74,)
  ✅ PRES                      : float32 | shape: (74, 1291)
  ✅ TEMP                      : float32 | shape: (74, 1291)
  ✅ PSAL                      : float32 | shape: (74, 1291)
  ✅ POSITIONING_SYSTEM        : object | shape: (74,)
  ✅ PLATFORM_TYPE             : object | shape: (74,)
  ✅ DATA_CENTRE               : object | shape: (74,)
  ✅ PROJECT_NAME              : object | shape: (74,)
<xarray.Dataset> Size: 8MB
Dimensions:                       (N_PROF: 74, N_PARAM: 3, N_LEVELS: 1291,
                                   N_CALIB: 3, N_HISTORY: 0)
Dimensions without coordinates: N_PROF, N_PARAM, N_LEVELS, N_CALIB, N_HISTORY
Data variables: (12/64)
    DATA_TYPE                     object 8B ...
    FORMAT_VERSION                object 8

In [52]:
import pandas as pd
import requests
import os
import io

print("📥 BGC Index download ho raha hai...")

url = "https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt"
r = requests.get(url, timeout=120)

# Skip comment lines (# se shuru hoti hain)
content = r.text
lines = [l for l in content.split('\n') if not l.startswith('#')]
clean = '\n'.join(lines)

# Pandas mein load karo
df = pd.read_csv(io.StringIO(clean))

print(f"✅ Total BGC profiles worldwide: {len(df)}")
print(f"\nColumns: {list(df.columns)}")

# Sirf Indian Ocean filter karo
indian = df[df['ocean'] == 'I']
print(f"\n🇮🇳 Indian Ocean BGC profiles: {len(indian)}")

# INCOIS floats filter karo
incois = indian[indian['institution'] == 'IN']
print(f"🏛️  INCOIS BGC profiles: {len(incois)}")

# Parameters dekho
print("\n🧪 Available BGC Parameters:")
print(indian['parameters'].value_counts().head(10))

📥 BGC Index download ho raha hai...
✅ Total BGC profiles worldwide: 384020

Columns: ['file', 'date', 'latitude', 'longitude', 'ocean', 'profiler_type', 'institution', 'parameters', 'parameter_data_mode', 'date_update']

🇮🇳 Indian Ocean BGC profiles: 68978
🏛️  INCOIS BGC profiles: 14203

🧪 Available BGC Parameters:
parameters
PRES TEMP PSAL DOXY                                                                                                                      21640
PRES TEMP PSAL DOXY CHLA BBP700                                                                                                          12885
PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP700 PH_IN_SITU_TOTAL NITRATE                                                                7327
PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IRRADIANCE412 DOWN_IRRADIANCE490 DOWNWELLING_PAR CHLA CHLA_FLUORESCENCE BBP700 CDOM           3561
PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP700                                                   

In [58]:
import requests
import pandas as pd
from io import StringIO

url = "https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt"

print("Downloading BGC index file...")
r = requests.get(url, timeout=120, stream=True)
print(f"Status: {r.status_code}")

if r.status_code == 200:
    # Pehli 50 lines dekho
    lines = r.text.split('\n')
    print(f"\nTotal lines: {len(lines)}")
    print("\nPehli 10 lines:")
    for line in lines[:10]:
        print(line)

Rich BGC profiles (CHLA+DOXY): 12974
Sample files to download: 15


C:\Users\HP\AppData\Local\Temp\ipykernel_12296\3031596561.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rich_bgc['date'] = pd.to_datetime(rich_bgc['date'], format='%Y%m%d%H%M%S')
C:\Users\HP\AppData\Local\Temp\ipykernel_12296\3031596561.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rich_bgc['year'] = rich_bgc['date'].dt.year


✅ SD2902120_250.nc — 0.0 MB
✅ SD2902120_251.nc — 0.0 MB
✅ SD2902120_252.nc — 0.0 MB
✅ SD2902209_197.nc — 0.1 MB
✅ SD2902209_198.nc — 0.1 MB
✅ SD2902209_199.nc — 0.1 MB
✅ SD2902209_234.nc — 0.1 MB
✅ SD2902209_235.nc — 0.1 MB
✅ SD2902209_236.nc — 0.1 MB
✅ SD2902263_308.nc — 0.1 MB
✅ SD2902263_309.nc — 0.1 MB
✅ SD2902263_310.nc — 0.1 MB
✅ SD2902272_208.nc — 0.1 MB
✅ SD2902272_209.nc — 0.1 MB
✅ SD2902272_210.nc — 0.1 MB

🎉 BGC Sample Download Complete!
✅ 15/15 files downloaded


In [1]:
import requests
import os
import xarray as xr
import numpy as np
from datetime import datetime, timedelta

CACHE_DIR = "argo_cache"
BASE_URL  = "https://data-argo.ifremer.fr/geo/indian_ocean/"
os.makedirs(CACHE_DIR, exist_ok=True)

# ===============================
# NEW DATE LOGIC → Every 10 Days
# ===============================

sample_dates = []

years = [2024, 2025, 2026]

for year in years:
    for month in range(1, 13):

        d = datetime(year, month, 1)

        while d.month == month:
            sample_dates.append(d)
            d += timedelta(days=10)

print(f"Total dates: {len(sample_dates)}")
print("Downloading...\n")

success = 0
failed  = 0

for date in sample_dates:
    filename   = date.strftime("%Y%m%d") + "_prof.nc"
    year       = date.strftime("%Y")
    month      = date.strftime("%m")
    cache_path = os.path.join(CACHE_DIR, filename)

    # Already exists? Skip
    if os.path.exists(cache_path):
        print(f"⏭️  Already exists: {filename}")
        success += 1
        continue

    url = f"{BASE_URL}{year}/{month}/{filename}"

    try:
        r = requests.get(url, timeout=120, stream=True)
        if r.status_code == 200:
            with open(cache_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    f.write(chunk)
            size = os.path.getsize(cache_path) / (1024*1024)
            print(f"✅ {filename} — {size:.1f} MB")
            success += 1
        else:
            print(f"❌ {filename} — Status {r.status_code}")
            failed += 1
    except Exception as e:
        print(f"⚠️  {filename} — Error: {e}")
        failed += 1

# Summary
total_size = sum(
    os.path.getsize(os.path.join(CACHE_DIR, f)) 
    for f in os.listdir(CACHE_DIR) 
    if f.endswith('.nc')
) / (1024*1024)

print(f"""
🎉 Download Complete!
━━━━━━━━━━━━━━━━━━━━
✅ Success : {success} files
❌ Failed  : {failed} files
💾 Total   : {total_size:.1f} MB
""")

Total dates: 129
Downloading...

⏭️  Already exists: 20240101_prof.nc
✅ 20240111_prof.nc — 3.8 MB
⏭️  Already exists: 20240121_prof.nc
⏭️  Already exists: 20240131_prof.nc
⏭️  Already exists: 20240201_prof.nc
✅ 20240211_prof.nc — 7.0 MB
✅ 20240221_prof.nc — 5.5 MB
✅ 20240301_prof.nc — 4.1 MB
⏭️  Already exists: 20240311_prof.nc
✅ 20240321_prof.nc — 4.1 MB
⏭️  Already exists: 20240331_prof.nc
⏭️  Already exists: 20240401_prof.nc
✅ 20240411_prof.nc — 4.8 MB
⏭️  Already exists: 20240421_prof.nc
⏭️  Already exists: 20240501_prof.nc
⏭️  Already exists: 20240511_prof.nc
⏭️  Already exists: 20240521_prof.nc
⏭️  Already exists: 20240531_prof.nc
⏭️  Already exists: 20240601_prof.nc
⏭️  Already exists: 20240611_prof.nc
⏭️  Already exists: 20240621_prof.nc
⏭️  Already exists: 20240701_prof.nc
⏭️  Already exists: 20240711_prof.nc
⏭️  Already exists: 20240721_prof.nc
⏭️  Already exists: 20240731_prof.nc
⏭️  Already exists: 20240801_prof.nc
⏭️  Already exists: 20240811_prof.nc
⏭️  Already exists: 20

In [ ]:
import requests
import os
import time
from datetime import datetime, timedelta

# ===============================
# CONFIG
# ===============================
CACHE_DIR = "argo_cache"
BASE_URL  = "https://data-argo.ifremer.fr/geo/indian_ocean/"

# Date range (change if needed)
START_DATE = datetime(2021, 1, 1)
END_DATE   = datetime(2026, 12, 31)

# ===============================
# SETUP
# ===============================
os.makedirs(CACHE_DIR, exist_ok=True)

print("📅 Generating daily dates...\n")

# Generate daily dates
sample_dates = []
d = START_DATE
while d <= END_DATE:
    sample_dates.append(d)
    d += timedelta(days=1)

print(f"Total dates: {len(sample_dates)}")
print("🚀 Starting download...\n")

# ===============================
# DOWNLOAD LOOP
# ===============================
success = 0
failed  = 0
skipped = 0

for date in sample_dates:
    filename   = date.strftime("%Y%m%d") + "_prof.nc"
    year       = date.strftime("%Y")
    month      = date.strftime("%m")
    cache_path = os.path.join(CACHE_DIR, filename)

    # Skip if already downloaded
    if os.path.exists(cache_path):
        print(f"⏭️  Skipped (exists): {filename}")
        skipped += 1
        continue

    url = f"{BASE_URL}{year}/{month}/{filename}"

    try:
        r = requests.get(url, timeout=120, stream=True)

        if r.status_code == 200:
            with open(cache_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)

            size = os.path.getsize(cache_path) / (1024*1024)
            print(f"✅ Downloaded: {filename} ({size:.1f} MB)")
            success += 1

        elif r.status_code == 404:
            print(f"❌ Not found: {filename}")
            failed += 1

        else:
            print(f"⚠️ Error {r.status_code}: {filename}")
            failed += 1

    except Exception as e:
        print(f"⚠️ Exception: {filename} → {e}")
        failed += 1

    # 🔥 Important: avoid server overload
    time.sleep(1)

# ===============================
# SUMMARY
# ===============================
total_size = sum(
    os.path.getsize(os.path.join(CACHE_DIR, f))
    for f in os.listdir(CACHE_DIR)
    if f.endswith(".nc")
) / (1024 * 1024)

print("\n🎉 Download Complete!")
print("━━━━━━━━━━━━━━━━━━━━")
print(f"✅ Success : {success} files")
print(f"❌ Failed  : {failed} files")
print(f"⏭️ Skipped : {skipped} files")
print(f"💾 Total   : {total_size:.1f} MB")

📅 Generating daily dates...

Total dates: 2191
🚀 Starting download...

⏭️  Skipped (exists): 20210101_prof.nc
✅ Downloaded: 20210102_prof.nc (4.4 MB)
✅ Downloaded: 20210103_prof.nc (4.4 MB)
